# 04 - Modeling, Validation, and Hyperparameter Tuning
Primary metric is MAE. Test data is touched only once at the end.

In [1]:
from pathlib import Path
import sys,warnings
warnings.filterwarnings("ignore")
ROOT=Path.cwd(); ROOT=ROOT.parent if ROOT.name=="notebooks" else ROOT
sys.path.insert(0,str(ROOT)) if str(ROOT) not in sys.path else None
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils import *
ensure_dirs(); pd.set_option("display.max_columns",100)
print("Project root:",ROOT)
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
import joblib

Project root: /Users/sangeetasarker/Downloads/cse437-used-car-price-prediction-GROUP_NUMBER


In [2]:
df=pd.read_csv(MODEL_DATA_PATH,low_memory=False); X=df[FINAL_FEATURES]; y=df.price; Xtr,Xtmp,ytr,ytmp=train_test_split(X,y,test_size=.2,random_state=RANDOM_STATE); Xv,Xte,yv,yte=train_test_split(Xtmp,ytmp,test_size=.5,random_state=RANDOM_STATE); print(len(Xtr),len(Xv),len(Xte))

96000 12000 12000


In [3]:
cat=Pipeline([("imputer",SimpleImputer(strategy="constant",fill_value="missing")),("onehot",OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=25,sparse_output=True))]); pre=ColumnTransformer([("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())]),NUMERIC_FEATURES),("cat",cat,CATEGORICAL_FEATURES)]); baseline=DummyRegressor(strategy="median").fit(Xtr,ytr); print("Baseline val",regression_metrics(yv,baseline.predict(Xv)))

Baseline val {'MAE': 11440.466583333333, 'RMSE': 15651.602105066859, 'R2': -0.053256867459116286, 'MAPE_percent': 131.55864571438482}


In [4]:
ridge=Pipeline([("pre",pre),("model",Ridge())]); gs=GridSearchCV(ridge,{"model__alpha":[.1,1,10,50,100]},cv=3,scoring="neg_mean_absolute_error",n_jobs=-1,return_train_score=True).fit(Xtr,ytr); pd.DataFrame(gs.cv_results_).to_csv(RESULTS_DIR/"ridge_search.csv",index=False); print(gs.best_params_,-gs.best_score_)

{'model__alpha': 50} 6158.517565079233


In [5]:
rfpre=ColumnTransformer([("num",Pipeline([("imputer",SimpleImputer(strategy="median"))]),NUMERIC_FEATURES),("cat",cat,CATEGORICAL_FEATURES)]); rf=Pipeline([("pre",rfpre),("model",RandomForestRegressor(random_state=RANDOM_STATE,n_jobs=-1))]); idx=Xtr.sample(min(len(Xtr),TUNING_MAX_ROWS),random_state=RANDOM_STATE).index; space={"model__n_estimators":[80,120,180],"model__max_depth":[None,12,20,30],"model__min_samples_split":[2,5,10],"model__min_samples_leaf":[1,2,4],"model__max_features":[1.0,"sqrt",.5]}; rs=RandomizedSearchCV(rf,space,n_iter=10,cv=3,scoring="neg_mean_absolute_error",random_state=RANDOM_STATE,n_jobs=-1,return_train_score=True).fit(Xtr.loc[idx],ytr.loc[idx]); pd.DataFrame(rs.cv_results_).to_csv(RESULTS_DIR/"rf_search.csv",index=False); print(rs.best_params_,-rs.best_score_)

{'model__n_estimators': 80, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 0.5, 'model__max_depth': 30} 3772.2106894662925


In [6]:
rfbest=rf.set_params(**rs.best_params_).fit(Xtr,ytr); val={"BaselineMedian":regression_metrics(yv,baseline.predict(Xv)),"Ridge":regression_metrics(yv,gs.best_estimator_.predict(Xv)),"RandomForest":regression_metrics(yv,rfbest.predict(Xv))}; display(pd.DataFrame(val).T)

,MAE,RMSE,R2,MAPE_percent
BaselineMedian,11440.466583,15651.602105,-0.053257,131.558646
Ridge,6291.205530,9878.345446,0.580449,95.073148
RandomForest,3325.524175,6646.263959,0.810079,53.196880


In [7]:
models={"BaselineMedian":baseline,"Ridge":gs.best_estimator_,"RandomForest":rfbest}; mets={}; out=Xte.copy(); out["actual_price"]=yte.values
for name,m in models.items():
 p=m.predict(Xte); out[f"pred_{name}"]=p; mets[name]=regression_metrics(yte,p)
display(pd.DataFrame(mets).T); pd.DataFrame(mets).T.to_csv(RESULTS_DIR/"test_metrics.csv"); out.to_csv(RESULTS_DIR/"test_predictions.csv",index=False); joblib.dump(gs.best_estimator_,MODELS_DIR/"ridge.joblib",compress=3); joblib.dump(rfbest,MODELS_DIR/"random_forest.joblib",compress=3); save_json({"split":{"train":len(Xtr),"validation":len(Xv),"test":len(Xte)},"primary_metric":"MAE","validation_metrics":val,"test_metrics":mets,"ridge_best_params":gs.best_params_,"rf_best_params":rs.best_params_},RESULTS_DIR/"model_results.json")

,MAE,RMSE,R2,MAPE_percent
BaselineMedian,11272.798250,15211.789275,-0.052855,133.468313
Ridge,6213.528505,9609.489446,0.579846,97.248188
RandomForest,3247.738759,6137.037022,0.828634,50.388304
